## 1. Current work directory

In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import os
print(os.getcwd())

In [ ]:
import sys, numpy as np
print("Python:", sys.executable)
print("NumPy:", np.__version__, np.__file__)

## 2. Import packages and define input/output path 

In [ ]:
import glob
import os
import time
import pickle
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from interface_analyzer import analyze_cfm, plot_cfm_k2_single
from interface_analyzer import PTMModifier, analyze_by_custom_modifier, CSPModifier

SAVE_DIR = LOCAL_OUTPUT_DIR

## 3. Postprocess CFG files, output results.
### Output Data Structure (`cfg_post.pkl`)

The `cfg_post.pkl` file stores the intermediate results generated by the parallel processing script (`Process.py`). This file is the primary input for the Capillary Fluctuation Method (CFM) analysis (`analyze_cfm_ptm`).

The file contains a dictionary where keys correspond to the frame ID (or snapshot number) of the molecular dynamics configuration. Each value is a dictionary containing the extracted physical data needed for fluctuation analysis:

| Data Field | Description | Importance for CFM |
| :--- | :--- | :--- |
| **`h_upper`** & **`h_lower`** | The 1D arrays defining the height profile $h(x)$ of the solid-liquid interfaces (upper and lower), determined using the Brown maximization method on the binned order parameter data. | **CRITICAL:** These arrays are Fourier-transformed to calculate the mean-squared amplitude $\langle|A(k)|^2\rangle$. |
| **`cell`** | The periodic boundary condition (PBC) matrix from OVITO, containing the box dimensions ($L_x, L_y, L_z$) required for calculating the wave vectors ($k$) and the prefactor $k_B T / (L_x L_y)$. | **CRITICAL:** Provides the necessary geometric context for the analysis. |
| **`x`**, **`z`** | Coordinates of the bin centers along the $x$ and $z$ directions, respectively. | Used for spatial reference and calculating bin width/FFT grid setup. |
| **`M`** | The raw 2D binned data of the order parameter (e.g., Centrosymmetry or PTM Solid Flag) used to identify the interface location. | Useful for debugging and verifying the quality of the phase identification. |

The downstream function `analyze_cfm_ptm` reads these dictionary values, aggregates the `h_upper` and `h_lower` profiles across all snapshots, and performs the Fourier analysis.

In [ ]:
MAX_WORKERS = 8 # Adjust based on your system CPU cores

# --- Global Modifier Instance ---
# Instantiate the modifier; parameters can be fixed here
# ptm_modifier = PTMModifier(binsx=150, rmsd_max=0.15)
modifier = PTMModifier(binsx=150, rmsd_max=0.15)
SAVE_PATH_PKL = SAVE_DIR / "cfg_post_PTM_equilfile.pkl"
DIR_PATH = CFG_DIR
# --- 1. Collect all cfg files ---
files = sorted(glob.glob(os.path.join(DIR_PATH, "cfg.Al_100_010.*")),
               key=lambda f: int(f.split(".")[-1])) # Assumes file ends with step ID

def worker(cfg_path):
    """
    Wrapper function for parallel execution.
    Uses the globally defined ptm_modifier instance to run the analysis.
    """
    # Use the new analyze_by_custom_modifier function to run
    # The analyze_by_PTM wrapper is still available, you could also use analyze_by_PTM(cfg_path, ...)
    return str(cfg_path), analyze_by_custom_modifier(str(cfg_path), modifier)

results_all = {}  # {frame_id: result_dict}

# --- 2. Parallel execution ---
start = time.time()
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(worker, f): f for f in files}

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Processing CFG files"):
        fname, res = fut.result()
        # Extract frame ID from the filename
        try:
            frame_id = int(fname.split(".")[-1])
        except ValueError:
            frame_id = fname # Use full name if ID parsing fails

        results_all[frame_id] = res

end = time.time()
print(f"Total processing time: {end - start:.2f} seconds")

# --- 3. Save the results ---
with open(SAVE_PATH_PKL, "wb") as f:
    pickle.dump(results_all, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Results saved to: {SAVE_PATH_PKL}")

## 4. Analysis results and plot k and k^2
### Function: `plot_cfm_k2_single(filename, ...)`

This function is designed to analyze the final Capillary Fluctuation Method (CFM) data (which should be stored in a `.dat` file containing $k^2$ vs $k_B T / (L_x L_y \langle|A(k)|^2\rangle)$). The slope of this linear fit directly relates to the interface stiffness ($\tilde{\gamma}$).

#### Key Functionality:

1.  **Optimal Range Selection:** The function iteratively fits the low-$k^2$ data points and selects the subset of points ($n$) that yields the **maximum Coefficient of Determination ($R^2$)**. This provides an objective measure for determining the most linear region of the CFM spectrum.
2.  **Linear Fitting:** Performs a linear fit ($y = m x + b$ or $y = m x$ if `through_origin=True`) on the selected data range.
3.  **Visualization:** Generates a plot showing all data points, highlighting the points used for the best fit, and displaying the resulting fit line.

#### Key Parameters:

| Parameter | Description |
| :--- | :--- |
| **`filename`** | Path to the input `.dat` file (must contain $k^2$, $\text{Ak}_{min}$, and $\text{Ak}_{max}$). |
| **`k2_min`** | Minimum $k^2$ value to consider for the fit. Filters out the first point which is usually zero. |
| **`min_points`** | Minimum number of data points required to perform the linear fit. |
| **`L_min_interface`** | Defines the maximum $k^2$ cutoff based on the expected minimum interface width ($\sim (2\pi / (L_{\text{min}} \cdot a))^2$). |
| **`through_origin`** | If `True`, forces the linear regression line to pass through the origin ($b=0$). |

#### Return Value:

Returns a dictionary containing the calculated fit results, including the calculated `slope` (interface stiffness), `intercept`, maximum `r2`, and the $k^2$ range (`k2_min_used`, `k2_max_used`).

In [ ]:
SAVE_DIR = FULL_DATA_ROOT / "111_1-21"
Path_pkl = SAVE_DIR / "111_1-21_cfg_post_orientation_grid_2_5_d_6.pkl"

# ### Step 2: Analyze and Plot CFM
TEMPERATURE_K = 932.6
LATTICE_CONST_A = 4.134
# --- 1. Run the CFM analysis ---
results_ptm = analyze_cfm(
    pickle_path=Path_pkl,
    T=TEMPERATURE_K,
    a=LATTICE_CONST_A,
    use_pchip=True, 
    pchipres=10000,
    show_plot=True
)

# --- 2. Save the final k^2 data for linear fitting ---
OUTPUT_BASE = SAVE_DIR / "ptm_cfm_output"

k2 = results_ptm["k2"]
Ak_min = results_ptm["Ak_min"]
Ak_max = results_ptm["Ak_max"]

# Data matrix (k^2, Ak_min, Ak_max)
intdata2 = np.c_[k2, Ak_min, Ak_max]

np.savetxt(
    str(OUTPUT_BASE) + "_k2.dat",
    intdata2,
    fmt="%.8e",
    header="k^2 Ak_min Ak_max"
)
print(f"CFM data saved to: {OUTPUT_BASE}_k2.dat")

# ### Step 3: Linear Fit and Plot (for $\gamma$)
res_fit = plot_cfm_k2_single(
            str(OUTPUT_BASE) + "_k2.dat",
            k_range=[0.07, 0.165],
            k2_min=5.0e-3,
            L_min_interface=4,
            min_points=6,
            through_origin=True,
            xlim=[0, 0.05],
            ylim=[0,0.6e-19]
        )

print("\n--- Linear Fit Results ---")
print(res_fit)

In [ ]:
Path_ptm_pkl = SAVE_DIR / "100_010_cfg_post_orientation_grid_1_5_ang.pkl"

## 5. Check the volume fraction of solid phase
CFM needs a relatively stable interface position, therefore the fraction of solid phase should be a constant.

In [ ]:
from matplotlib import pyplot as plt
with open(Path_ptm_pkl, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)
frames = sorted(results_all.keys())
mean_upper = [results_all[i]["h_upper"].mean() for i in frames]
mean_lower = [results_all[i]["h_lower"].mean() for i in frames]
solid = np.asarray(mean_upper) - np.asarray(mean_lower)

# Plot
plt.figure(figsize=(6,4))
plt.plot(frames, solid, 'o-', lw=1.5, markersize=4)
plt.xlabel("Frame index (timestep)")
plt.ylabel("Solid phase length (Å)")
plt.title("Evolution of solid-liquid interface position")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(solid,bins=20,label="100_010")
plt.title("Histogram of Solid Volume")
plt.legend()
plt.show()

In [ ]:
Phase = results_all[3000000]["M"]

plt.figure(figsize=(6, 5))
plt.imshow(Phase, cmap='viridis', origin='lower')
plt.colorbar(label='Phase Value')
plt.title('Phase Heatmap')
plt.xlabel('X index')
plt.ylabel('Y index')
plt.show()

In [ ]:
Boundary = results_all[3000000]["h_lower"]
plt.plot(Boundary)
plt.xlabel('X index')
plt.ylabel('Y index')
plt.show()

In [ ]:
import numpy as np

# 1. Collect and sort all timesteps to keep the time series ordered
timesteps = sorted(results_all.keys())

# 2. Use a list comprehension to extract h_lower (or h_upper) for each frame
# Assume each results_all[ts]["h_lower"] is an array or Series of length N_bins
h_lower_list = [results_all[ts]["h_lower"] for ts in timesteps]

# 3. Convert the list to a 2D NumPy array with shape (N_frames, N_bins)
h_lower_series = np.array(h_lower_list)

# 4. Check the shape
print(f"Number of frames (N_frames): {h_lower_series.shape[0]}")
print(f"Number of bins (N_bins): {h_lower_series.shape[1]}")

# This array can now be passed to the analysis function
# mu_slope = calculate_fluctuation_kinetics(h_lower_series, dt=..., Lx=...)

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

def calculate_advanced_mu(h_series, dt, Lx, Tm, latent_heat_vol, stiffness):
    """
    Compute the kinetic coefficient mu following [cite: 448-472], keeping the units self-consistent.
    
    Recommended unit system (LAMMPS metal units):
    --------------------------------
    h_series        : np.array (N_frames, N_bins), unit: [Angstrom]
    dt              : frame interval, unit: [ps]
    Lx              : interface box length, unit: [Angstrom]
    Tm              : melting point, unit: [K]
    latent_heat_vol : latent heat per unit volume (L), unit: [eV/Angstrom^3]
                      Note: if the input data are in J/m^3, divide by 1.60218e28
    stiffness       : interface stiffness (gamma + gamma''), unit: [eV/Angstrom^2]
                      Note: if the input data are in mJ/m^2 (or erg/cm^2), divide by 1.60218e3
    
    Returns:
    --------------------------------
    mu_SI           : kinetic coefficient, unit: [m/(s·K)]
    """
    
    # 1. Compute the Gibbs-Thomson coefficient Gamma (unit: A·K) [cite: 458]
    gamma_gt_MD = (Tm * stiffness) / latent_heat_vol
    
    n_frames, n_bins = h_series.shape
    k_vals = 2 * np.pi * np.fft.rfftfreq(n_bins, d=Lx/n_bins) # unit: A^-1
    amplitudes = np.fft.rfft(h_series - np.mean(h_series, axis=1, keepdims=True), axis=1, norm="forward")
    
    tau_list = []
    k_fit_list = []
    
    def normalized_decay_model(t, tau):
        return 1 - np.exp(-t / tau) # 

    max_lag = n_frames // 5
    time_lags = np.arange(max_lag) * dt # unit: ps

    for i in range(1, len(k_vals)):
        A_k = amplitudes[:, i]
        mean_sq_A = np.mean(np.abs(A_k)**2)
        
        lhs = []
        for lag in range(max_lag):
            diff_sq = np.abs(A_k[lag:] - A_k[:-lag if lag > 0 else None])**2
            lhs.append(np.mean(diff_sq) / (2 * mean_sq_A))
        lhs = np.array(lhs)
        
        try:
            # Fit tau (ps)
            popt, _ = curve_fit(normalized_decay_model, time_lags[lhs > 0.1], lhs[lhs > 0.1], p0=[10.0])
            tau_list.append(popt[0])
            k_fit_list.append(k_vals[i])
        except:
            continue

    k_fit = np.array(k_fit_list)
    tau_fit_ps = np.array(tau_list)
    
    # 2. Extract mu_k (SIunit: m/s/K) [cite: 451, 469]
    inv_tau = 1.0 / tau_fit_ps
    rhs_for_mu = gamma_gt_MD * (k_fit**2)
    mu_MD, _ = np.polyfit(rhs_for_mu, inv_tau, 1)
    mu_SI = mu_MD * 100.0 

    # --- 3. Plot the results (matching the coordinate range in the manuscript) ---
    # Convert tau to ns before plotting so that it falls in the 0.0008-0.2 range 
    tau_fit_ns = tau_fit_ps / 1000.0 
    
    plt.figure(figsize=(8, 6))
    
    # Plot simulation points
    plt.loglog(k_fit, tau_fit_ns, 'o', color='tab:blue', label='Simulation Data', markersize=7)
    
    # Plot the fitted line: tau(ns) = 1 / (mu_MD * Gamma * k^2 * 1000)
    tau_theory_ns = 1.0 / (mu_MD * gamma_gt_MD * k_fit**2 * 1000.0)
    plt.loglog(k_fit, tau_theory_ns, '-', color='tab:orange', 
               label=f'Fit: $\\mu_k$ = {mu_SI:.3f} m/s/K')
    
    # Set the requested plotting range
    plt.xlim(0.02, 0.4)   # x-axis range: 0.02 to 0.4 [A^-1]
    plt.ylim(0.0008, 0.2) # y-axis range: 0.0008 to 0.2 [ns]
    
    plt.xlabel(r'$k$ ($\AA^{-1}$)', fontsize=12)
    plt.ylabel(r'$\tau$ (ns)', fontsize=12)
    plt.title('Kinetic Coefficient Fitting (Fluctuation Spectra)', fontsize=13)
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    return mu_SI

def calculate_kinetic_anisotropy(mu_max, mu_min):
    """Compute the kinetic anisotropy parameter epsilon_k """
    return (mu_max - mu_min) / (mu_max + mu_min)

In [ ]:
latent_heat_vol_MD = 9.4e8 / 1.60218e11 # J/m^3 => ev/A^3
stiffness_MD = 101 / 1.60218e4 # mJ/m^2 => ev/A^2
mu_k = calculate_advanced_mu(h_lower_series, dt=2, Lx=584.18, Tm=907, latent_heat_vol=latent_heat_vol_MD, stiffness=stiffness_MD)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# --- Parameter settings; adjust for the simulation being analyzed ---
Lx = 584.18          # Box length in the X direction [Angstrom]
dt = 2.0              # trajectory output interval [ps]
Tm = 907.0            # melting point [K]
L_vol = latent_heat_vol_MD       # latent heat per unit volume [eV/A^3]
stiffness = stiffness_MD   # interface stiffness [eV/A^2]

# --- 1. Merge and extract data ---
timesteps = sorted(results_all.keys())
h_upper_all = np.array([results_all[ts]["h_upper"] for ts in timesteps])
h_lower_all = np.array([results_all[ts]["h_lower"] for ts in timesteps])
interfaces = [h_upper_all, h_lower_all]

N_frames, N_bins = h_upper_all.shape
k_vals = 2 * np.pi * np.fft.rfftfreq(N_bins, d=Lx/N_bins) # [A^-1]

# Perform the Fourier transform
amplitudes_list = [np.fft.rfft(h - np.mean(h, axis=1, keepdims=True), axis=1, norm="forward") for h in interfaces]

print(f"Data loaded: {N_frames} frames, {N_bins} bins. Extracted Fourier amplitudes for the two interfaces.")

In [ ]:
# --- 2. Fit settings ---
max_lag = N_frames // 5
time_lags_ns = np.arange(max_lag) * dt / 1000.0
tau_results = []
k_results = []

def decay_model(t, tau):
    return 1 - np.exp(-t / tau) # [cite: 451]

plt.figure(figsize=(6, 6))

# Loop over the specified index range: 5 to 13 (inclusive)
for i in range(7, 14):
    # Joint averaging: combine statistics from the two interfaces
    sum_diff_sq = np.zeros(max_lag)
    sum_total_power = 0
    
    for amp in amplitudes_list:
        A_k = amp[:, i]
        sum_total_power += 2 * np.mean(np.abs(A_k)**2) # 2 * <|A(k,0)|^2> [cite: 451, 483]
        for lag in range(max_lag):
            diff = np.abs(A_k[lag:] - A_k[:-lag if lag > 0 else None])**2
            sum_diff_sq[lag] += np.mean(diff) # <|A(k,t)-A(k,0)|^2> [cite: 451, 483]
            
    # Compute the normalized correlation function LHS
    lhs = sum_diff_sq / sum_total_power
    is_too_high = (lhs >= 0.96)
    has_reached_cutoff = np.maximum.accumulate(is_too_high)
    # Set the fitting mask; use only points between 0.7 and 0.96
    fit_mask = (~has_reached_cutoff) & (lhs > 0.7)
    not_fit_mask = ~fit_mask
    if np.sum(fit_mask) > 3: # Ensure that enough points are available for fitting
        try:
            # Fit and extract tau [ns] [cite: 468]
            popt, _ = curve_fit(decay_model, time_lags_ns[fit_mask], lhs[fit_mask], p0=[0.01])
            tau_val = popt[0]
            tau_results.append(tau_val)
            k_results.append(k_vals[i])
            
            # Plot to inspect fit quality
            color = plt.cm.plasma((i-5)/9)
            plt.scatter(time_lags_ns[not_fit_mask], lhs[not_fit_mask], 
                        s=10, facecolors='none', edgecolors=color, alpha=0.3)

            # 2. Plot the selected points as filled markers 
            plt.scatter(time_lags_ns[fit_mask], lhs[fit_mask], 
                        s=15, color=color, alpha=0.8)
            plt.plot(time_lags_ns, decay_model(time_lags_ns, tau_val), color=color, 
                     label=f'k={k_vals[i]:.4f}, $\\tau$={tau_val:.4f}ns')
        except:
            print(f"Index {i} (k={k_vals[i]:.4f}) fit failed")

plt.axhline(0.7, color='gray', linestyle=':', alpha=0.5)
plt.axhline(0.96, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Time (ns)')
plt.ylabel('Normalized Correlation Function')
plt.title('110[1-10]')
plt.xlim(-0.002, 0.1)
plt.legend(loc='best')
plt.grid(True, alpha=0.2)
# plt.savefig("110_1-10_decay.svg",format="svg")
plt.show()

In [ ]:
# --- 3. Extract the kinetic coefficient mu ---
k_array = np.array(k_results)
tau_array = np.array(tau_results)
gamma_gt = (Tm * stiffness) / L_vol # [A*K] [cite: 458]

# Linear fitting: 1/tau = (mu * Gamma) * k^2 -> mu = 1 / (tau * Gamma * k^2) [cite: 451, 452]
inv_tau = 1.0 / tau_array
rhs = gamma_gt * (k_array**2)
mu_MD, _ = np.polyfit(rhs, inv_tau, 1) # the slope is mu [A/(ns*K)]
mu_SI = mu_MD * 0.1 # Convert A/ns to m/s; result unit is m/s/K

print(f"Fitted kinetic coefficient mu: {mu_SI:.4f} m/s/K")

# --- Plot the results ---
plt.figure(figsize=(7, 6))
plt.loglog(k_array, tau_array, 'ks', label='MD Results (Combined)')

# Plot the reference line: tau = 1 / (mu * Gamma * k^2)
k_ref = np.linspace(0.02, 0.4, 100)
tau_ref = 1.0 / (gamma_gt * mu_MD * k_ref**2)
plt.loglog(k_ref, tau_ref, 'r-', label=f'Fit Line ($\mu$={mu_SI:.3f} m/s/K)')

plt.xlim(0.02, 0.4)   # range used in the manuscript [cite: 472]
plt.ylim(0.0008, 0.2) # range used in the manuscript [cite: 472]
plt.xlabel(r'$k$ ($\AA^{-1}$)')
plt.ylabel(r'$\tau$ (ns)')
plt.title('110[1-10]')
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.legend()
# plt.savefig("110_1-10_mu_k_fitting.svg",format="svg")
plt.show()

## 6. Variation of the slope with respect to the k-range
### Function: `analyze_cfm_fit_sensitivity(filename, ...)`

#### Interface Stiffness Sensitivity Analysis

The choice of the low-$k^2$ fitting range is often subjective and critical to the final calculated interface stiffness ($\tilde{\gamma}$). This function quantifies the robustness of the result by analyzing the fit parameters as a function of the data range.

It systematically iterates through all valid low-$k^2$ data subsets (starting from the minimum number of points, `min_points`, up to the maximum filtered dataset size) and performs a linear fit for each subset.

#### Key Output:

The function generates two plots:

1.  **Interface Stiffness (Slope $m$) vs. Number of $k^2$ Points ($n$):** Shows how sensitive the stiffness value is to the inclusion of higher-$k^2$ data points.
2.  **Coefficient of Determination ($R^2$) vs. Number of $k^2$ Points ($n$):** Identifies the range where the linear relationship is strongest (highest $R^2$).

#### Return Value:

Returns a dictionary containing arrays for:
* `n_points`: The number of data points used in each successive fit.
* `stiffness`: The slope ($m$) calculated for each fit.
* `r2`: The $R^2$ value calculated for each fit.

In [ ]:
from interface_analyzer import analyze_cfm_fit_sensitivity

fit_results = analyze_cfm_fit_sensitivity(
    csp_k2_path,
    k2_min=10.0e-4,
    min_points=5,
    L_min_interface=4,
    through_origin=True
)

# The plots will be displayed, and fit_results will contain arrays
# for 'n_points', 'stiffness', and 'r2'.
print("Stiffness values calculated:", fit_results['stiffness'])
csp_fit_output = "csp_cfm_fit_sensitivity.dat"
data_matrix = np.c_[fit_results['n_points'], fit_results['stiffness'], fit_results['r2']]

# 2. Define the header
header = "N_Points_Used | Stiffness_Slope | R2_Coefficient"
np.savetxt(
        csp_fit_output,
        data_matrix,
        fmt="%d %.8e %.8f",  # Format: integer, scientific notation, fixed float
        header=header,
        comments='# '        # Ensures header is commented out
    )

In [ ]:
from interface_analyzer import analyze_cfm_fit_sensitivity
fit_results = analyze_cfm_fit_sensitivity(
    ptm_k2_path,
    k2_min=10.0e-4,
    min_points=5,
    L_min_interface=4,
    through_origin=True
)

# The plots will be displayed, and fit_results will contain arrays
# for 'n_points', 'stiffness', and 'r2'.
print("Stiffness values calculated:", fit_results['stiffness'])
ptm_fit_output = "ptm_cfm_fit_sensitivity.dat"
data_matrix = np.c_[fit_results['n_points'], fit_results['stiffness'], fit_results['r2']]

# 2. Define the header
header = "N_Points_Used | Stiffness_Slope | R2_Coefficient"
np.savetxt(
        ptm_fit_output,
        data_matrix,
        fmt="%d %.8e %.8f",  # Format: integer, scientific notation, fixed float
        header=header,
        comments='# '        # Ensures header is commented out
    )